In [3]:
import os

for root, dirs, files in os.walk(".."):
    if "venv" in root or ".git" in root:
        continue
    for f in files:
        print(os.path.join(root, f))

../.DS_Store
../README.md
../.gitignore
../data/.DS_Store
../data/test.csv
../data/data_description.txt
../data/train.csv
../data/sample_submission.csv
../data/processed/train_clean.csv
../data/processed/experiment_log.csv
../data/processed/X_train_engineered.csv
../data/processed/test_clean.csv
../data/processed/y_train.csv
../data/processed/X_test_engineered.csv
../notebooks/05_model_tuning.ipynb
../notebooks/04_baseline_model.ipynb
../notebooks/.gitkeep
../notebooks/01_eda.ipynb
../notebooks/02_data_processing.ipynb
../notebooks/03_feature_engineering.ipynb
../notebooks/.ipynb_checkpoints/05_model_tuning-checkpoint.ipynb
../notebooks/.ipynb_checkpoints/02_data_processing-checkpoint.ipynb
../notebooks/.ipynb_checkpoints/01_eda-checkpoint.ipynb
../notebooks/.ipynb_checkpoints/05_model_tuning-checkpoint 2.ipynb
../notebooks/.ipynb_checkpoints/03_feature_engineering-checkpoint.ipynb
../notebooks/.ipynb_checkpoints/04_baseline_model-checkpoint.ipynb
../src/.gitkeep


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import optuna
import warnings
warnings.filterwarnings("ignore")

# Load feature-engineered data from Day 5/6
X = pd.read_csv("../data/processed/X_train_engineered.csv")
y = pd.read_csv("../data/processed/y_train.csv")
X_test = pd.read_csv("../data/processed/X_test_engineered.csv")

# y_train.csv loads as a DataFrame, convert to 1D array
y = y.iloc[:, 0].values if y.shape[1] == 1 else y.values.ravel()

# Test set Ids, needed later to build the submission file
test_clean = pd.read_csv("../data/processed/test_clean.csv")
test_ids = test_clean["Id"].values if "Id" in test_clean.columns else pd.read_csv("../data/test.csv")["Id"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("First 5 y values:", y[:5])

X shape: (1458, 306)
y shape: (1458,)
X_test shape: (1459, 306)
First 5 y values: [12.24769912 12.10901644 12.31717117 11.84940484 12.4292202 ]


In [5]:
N_FOLDS = 10
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def run_cv(model_fn, X, y, X_test, n_folds=N_FOLDS):
    """
    model_fn: a function that takes (X_train, y_train, X_val, y_val, X_test)
    and returns (fitted_model, val_pred, test_pred)
    Returns: oof predictions, averaged test predictions, per-fold scores, overall OOF RMSE
    """
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model, val_pred, test_pred = model_fn(X_tr, y_tr, X_val, y_val, X_test)
        oof[val_idx] = val_pred
        test_preds += test_pred / n_folds

        score = rmse(y_val, val_pred)
        fold_scores.append(score)
        print(f"Fold {fold+1}/{n_folds} RMSE: {score:.5f}")

    overall = rmse(y, oof)
    print(f"OOF RMSE: {overall:.5f}")
    return oof, test_preds, fold_scores, overall

In [6]:
def xgb_model_fn(X_tr, y_tr, X_val, y_val, X_test, params=None):
    default_params = dict(
        learning_rate=0.05,   # fixed learning rate for baseline
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        n_estimators=2000,
        random_state=42,
        n_jobs=-1,
    )
    if params:
        default_params.update(params)

    model = xgb.XGBRegressor(**default_params, early_stopping_rounds=100, eval_metric="rmse")
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    return model, val_pred, test_pred

xgb_oof, xgb_test, xgb_fold_scores, xgb_overall = run_cv(xgb_model_fn, X, y, X_test)

Fold 1/10 RMSE: 0.11076
Fold 2/10 RMSE: 0.12936
Fold 3/10 RMSE: 0.10846
Fold 4/10 RMSE: 0.10829
Fold 5/10 RMSE: 0.13508
Fold 6/10 RMSE: 0.10790
Fold 7/10 RMSE: 0.12333
Fold 8/10 RMSE: 0.11395
Fold 9/10 RMSE: 0.11478
Fold 10/10 RMSE: 0.09904
OOF RMSE: 0.11558


In [7]:
def lgb_model_fn(X_tr, y_tr, X_val, y_val, X_test, params=None):
    default_params = dict(
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        n_estimators=2000,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    if params:
        default_params.update(params)

    model = lgb.LGBMRegressor(**default_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    return model, val_pred, test_pred

lgb_oof, lgb_test, lgb_fold_scores, lgb_overall = run_cv(lgb_model_fn, X, y, X_test)

Fold 1/10 RMSE: 0.12405
Fold 2/10 RMSE: 0.13234
Fold 3/10 RMSE: 0.11801
Fold 4/10 RMSE: 0.10771
Fold 5/10 RMSE: 0.14173
Fold 6/10 RMSE: 0.11166
Fold 7/10 RMSE: 0.13291
Fold 8/10 RMSE: 0.12216
Fold 9/10 RMSE: 0.11342
Fold 10/10 RMSE: 0.09325
OOF RMSE: 0.12049


In [8]:
def cat_model_fn(X_tr, y_tr, X_val, y_val, X_test, params=None):
    default_params = dict(
        learning_rate=0.05,
        depth=6,
        subsample=0.8,
        iterations=2000,
        random_seed=42,
        verbose=False,
    )
    if params:
        default_params.update(params)

    model = CatBoostRegressor(**default_params, early_stopping_rounds=100)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    return model, val_pred, test_pred

cat_oof, cat_test, cat_fold_scores, cat_overall = run_cv(cat_model_fn, X, y, X_test)

Fold 1/10 RMSE: 0.09807
Fold 2/10 RMSE: 0.12285
Fold 3/10 RMSE: 0.10976
Fold 4/10 RMSE: 0.10639
Fold 5/10 RMSE: 0.12670
Fold 6/10 RMSE: 0.10638
Fold 7/10 RMSE: 0.12990
Fold 8/10 RMSE: 0.11099
Fold 9/10 RMSE: 0.11159
Fold 10/10 RMSE: 0.09297
OOF RMSE: 0.11214


In [9]:
import os
os.makedirs("../output/oof", exist_ok=True)

np.save("../output/oof/xgb_oof.npy", xgb_oof)
np.save("../output/oof/xgb_test.npy", xgb_test)
np.save("../output/oof/lgb_oof.npy", lgb_oof)
np.save("../output/oof/lgb_test.npy", lgb_test)
np.save("../output/oof/cat_oof.npy", cat_oof)
np.save("../output/oof/cat_test.npy", cat_test)

# Record each model's OOF RMSE for comparison later
summary = pd.DataFrame({
    "model": ["xgboost", "lightgbm", "catboost"],
    "oof_rmse": [xgb_overall, lgb_overall, cat_overall],
})
summary.to_csv("../output/oof/model_summary.csv", index=False)
print(summary.sort_values("oof_rmse"))

      model  oof_rmse
2  catboost  0.112144
0   xgboost  0.115580
1  lightgbm  0.120492
